## 1. Setup and Load Data

This stage examines whether state-level GDP provides additional explanatory information about loan default after controlling for traditional borrower and loan characteristics.

We compare a baseline logistic regression model using traditional credit, borrower, and loan characteristics with an augmented logistic regression model that additionally includes 2012 total state GDP.

In [1]:
# Import libraries for data preparation and statistical analysis
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load the final cleaned dataset from Stage 1
df = pd.read_csv('LendingClub_GDP_Final.csv')

# Check the dataset
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10447, 18)


,fico,dti,delinq_2yrs,inq_last_6mths,pub_rec,revol_util,total_acc,annual_inc,emp_length,home_ownership,loan_amnt,int_rate,term,grade,purpose,addr_state,state_gdp_2012,default
0,662.0,17.00,0.0,0.0,0.0,54.6%,15.0,53000.0,10+ years,MORTGAGE,2500.0,14.65%,36 months,C,home_improvement,GA,443566.1,0
1,732.0,8.21,0.0,0.0,0.0,26.1%,21.0,45600.0,10+ years,MORTGAGE,3000.0,7.90%,36 months,A,car,NV,127789.1,0
2,662.0,17.30,0.0,0.0,0.0,91.5%,13.0,76320.0,6 years,RENT,5600.0,19.22%,36 months,D,debt_consolidation,FL,768722.9,0
3,817.0,5.32,0.0,0.0,0.0,4.1%,6.0,65000.0,7 years,MORTGAGE,2800.0,7.62%,36 months,A,major_purchase,NY,1323400.8,0
4,702.0,4.23,0.0,2.0,0.0,21.3%,10.0,20700.0,< 1 year,OWN,1925.0,10.74%,36 months,B,home_improvement,AZ,268068.2,0


## 2. Define Variables for Statistical Analysis

To examine whether state-level GDP provides additional explanatory information for loan default, we define two logistic regression specifications.

**Baseline Model — Traditional Borrower and Loan Characteristics**

- FICO score (`fico`)
- Debt-to-income ratio (`dti`)
- Annual income (`annual_inc`)
- Revolving utilization (`revol_util`)
- Loan amount (`loan_amnt`)
- Loan term (`term`)

**Augmented Model — Traditional Characteristics + State GDP**

The augmented model includes all baseline variables plus:

- 2012 total state GDP (`state_gdp_2012`)

Using the same traditional variables in both models allows us to isolate whether adding state GDP provides additional explanatory information about loan default.

## 3. Prepare Variables for Logistic Regression

Selected numerical and categorical variables are prepared for regression analysis. Missing values are handled before model estimation, and categorical loan term is converted into a binary indicator.

In [2]:
# Convert revolving utilization from percentage strings into numeric percentage values
df['revol_util_num'] = (
    df['revol_util']
    .str.rstrip('%')
    .astype(float)
)

# Convert loan term into a binary indicator: 0 = 36 months; 1 = 60 months
df['term_60'] = (
    df['term']
    .str.strip()
    .eq('60 months')
    .astype(int)
)

In [3]:
# Define variables required for the statistical analysis
analysis_variables = [
    'fico',
    'dti',
    'annual_inc',
    'revol_util_num',
    'loan_amnt',
    'term_60',
    'state_gdp_2012',
    'default'
]

# Keep observations with complete information for the variables used in the logistic regression.
regression_data = (
    df[analysis_variables]
    .dropna()
    .copy()
)

print("Original observations:", len(df))
print("Regression observations:", len(regression_data))
print("Observations removed:", len(df) - len(regression_data))

Original observations: 10447
Regression observations: 10438
Observations removed: 9


**Note:** 9 observations are removed due to missing revolving utilization values. Since this represents only a very small portion of the dataset, the remaining 10,438 complete observations are used for the regression analysis.

## 4. Scale GDP for Interpretation

In [4]:
# Rescale state GDP for easier coefficient interpretation
# Original GDP is measured in millions of current dollars
# Dividing by 100,000 means a one-unit increase represents a $100 billion increase in total state GDP
regression_data['state_gdp_100b'] = (
    regression_data['state_gdp_2012'] / 100000
)

In [5]:
print("Original observations:", len(df))
print("Regression observations:", len(regression_data))
print("Observations removed:", len(df) - len(regression_data))

regression_data[
    [
        'fico',
        'dti',
        'annual_inc',
        'revol_util_num',
        'loan_amnt',
        'term_60',
        'state_gdp_100b',
        'default'
    ]
].describe()

Original observations: 10447
Regression observations: 10438
Observations removed: 9


,fico,dti,annual_inc,revol_util_num,loan_amnt,term_60,state_gdp_100b,default
count,10438.000000,10438.000000,1.043800e+04,10438.000000,10438.000000,10438.000000,10438.000000,10438.000000
mean,706.852989,15.374605,6.916127e+04,57.268548,13155.324296,0.189404,8.833601,0.170243
std,34.121897,6.972696,4.481035e+04,25.627366,8201.988169,0.391848,7.009169,0.375864
min,662.000000,0.000000,5.000000e+03,0.000000,1000.000000,0.000000,0.288936,0.000000
25%,682.000000,10.190000,4.200000e+04,39.200000,7000.000000,0.000000,3.345555,0.000000
50%,697.000000,15.480000,6.000000e+04,61.900000,11200.000000,0.000000,5.408824,0.000000
75%,727.000000,20.540000,8.400000e+04,78.100000,18000.000000,0.000000,13.234008,0.000000
max,847.500000,34.920000,1.233000e+06,97.900000,35000.000000,1.000000,21.440896,1.000000


## 5. Baseline Logistic Regression

The baseline logistic regression estimates the relationship between traditional borrower and loan characteristics and the probability of loan default.

The baseline model includes FICO score, debt-to-income ratio, annual income, revolving utilization, loan amount, and loan term. State GDP is excluded at this stage so that the model can later be compared with an augmented specification that includes state GDP.

The baseline logistic regression model can be expressed as:

$$
\log\left(\frac{P(\text{Default}=1)}
{1-P(\text{Default}=1)}\right)
=
\beta_0
+\beta_1(\text{FICO})
+\beta_2(\text{DTI})
+\beta_3(\text{Annual Income})
+\beta_4(\text{Revolving Utilization})
+\beta_5(\text{Loan Amount})
+\beta_6(\text{60-Month Term})
$$

This model serves as the baseline specification using only traditional borrower and loan characteristics. State GDP will be added separately in the augmented model to evaluate whether it provides additional explanatory information about loan default.

In [6]:
# Define the traditional borrower and loan characteristics
baseline_features = [
    'fico',
    'dti',
    'annual_inc',
    'revol_util_num',
    'loan_amnt',
    'term_60'
]

# Define the explanatory variables and target
X_baseline = regression_data[baseline_features]
y = regression_data['default']

# Add a constant term for the logistic regression intercept
X_baseline = sm.add_constant(X_baseline)

# Estimate the baseline logistic regression model
baseline_model = sm.Logit(y, X_baseline).fit()

# Display the regression results
print(baseline_model.summary())

Optimization terminated successfully.
         Current function value: 0.428298
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                10438
Model:                          Logit   Df Residuals:                    10431
Method:                           MLE   Df Model:                            6
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                 0.06131
Time:                        03:36:06   Log-Likelihood:                -4470.6
converged:                       True   LL-Null:                       -4762.6
Covariance Type:            nonrobust   LLR p-value:                6.618e-123
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              7.7943      0.818      9.531      0.000       6.191       9.397
fico             

**Finding:** The baseline logistic regression shows that FICO score, DTI, annual income, loan amount, and loan term are statistically associated with loan default. Higher FICO scores and annual income are associated with lower default risk, while higher DTI, larger loan amounts, and 60-month loan terms are associated with higher default risk. Revolving utilization is not statistically significant after controlling for the other variables (p = 0.873). The baseline model is statistically significant overall (LLR p-value < 0.001).

## 6. Augmented Logistic Regression with State GDP

The augmented logistic regression extends the baseline model by adding 2012 total state GDP. All traditional borrower and loan characteristics remain unchanged, allowing us to examine whether state GDP provides additional explanatory information about loan default after controlling for the baseline characteristics.

The augmented logistic regression model can be expressed as:

$$
\log\left(\frac{P(\text{Default}=1)}
{1-P(\text{Default}=1)}\right)
=
\beta_0
+\beta_1(\text{FICO})
+\beta_2(\text{DTI})
+\beta_3(\text{Annual Income})
+\beta_4(\text{Revolving Utilization})
+\beta_5(\text{Loan Amount})
+\beta_6(\text{60-Month Term})
+\beta_7(\text{State GDP})
$$

State GDP is measured in units of $100$ billion, so its coefficient represents the change in the log-odds of default associated with a $100$ billion increase in total state GDP, holding the other variables constant.

In [7]:
# Define the augmented feature set by adding state GDP
augmented_features = [
    'fico',
    'dti',
    'annual_inc',
    'revol_util_num',
    'loan_amnt',
    'term_60',
    'state_gdp_100b'
]

# Define the explanatory variables
X_augmented = regression_data[augmented_features]

# Add a constant term for the logistic regression intercept
X_augmented = sm.add_constant(X_augmented)

# Estimate the augmented logistic regression model
augmented_model = sm.Logit(y, X_augmented).fit()

# Display the regression results
print(augmented_model.summary())

Optimization terminated successfully.
         Current function value: 0.428276
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                10438
Model:                          Logit   Df Residuals:                    10430
Method:                           MLE   Df Model:                            7
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                 0.06136
Time:                        03:36:06   Log-Likelihood:                -4470.3
converged:                       True   LL-Null:                       -4762.6
Covariance Type:            nonrobust   LLR p-value:                5.448e-122
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              7.8157      0.818      9.551      0.000       6.212       9.420
fico             

 **Finding:** After controlling for traditional borrower and loan characteristics, 2012 total state GDP has a small negative coefficient (-0.0026), suggesting that borrowers in states with higher total GDP are estimated to have slightly lower default risk. However, the relationship is not statistically significant (p = 0.502), and the 95% confidence interval includes zero. Therefore, this model does not provide evidence that total state GDP provides statistically significant additional explanatory information about loan default after controlling for the baseline characteristics.

## 7. Odds Ratios and Confidence Intervals

Logistic regression coefficients are expressed in log-odds, which can be difficult to interpret directly. We therefore convert the augmented model coefficients into odds ratios.

An odds ratio greater than 1 indicates higher odds of default, while an odds ratio below 1 indicates lower odds of default, holding the other variables constant.

In [8]:
# Calculate odds ratios from the augmented logistic regression coefficients
odds_ratios = np.exp(augmented_model.params)

# Calculate 95% confidence intervals for the odds ratios
confidence_intervals = np.exp(augmented_model.conf_int())

# Combine results into one table
odds_ratio_table = pd.DataFrame({
    'Odds Ratio': odds_ratios,
    'CI Lower': confidence_intervals[0],
    'CI Upper': confidence_intervals[1],
    'P-value': augmented_model.pvalues
})

odds_ratio_table.round(4)

,Odds Ratio,CI Lower,CI Upper,P-value
const,2479.1778,498.5716,12327.8627,0.0000
fico,0.9864,0.9843,0.9886,0.0000
dti,1.0093,1.0011,1.0176,0.0266
annual_inc,1.0000,1.0000,1.0000,0.0000
revol_util_num,0.9998,0.9971,1.0025,0.8838
loan_amnt,1.0000,1.0000,1.0000,0.0000
term_60,2.1571,1.8829,2.4711,0.0000
state_gdp_100b,0.9974,0.9898,1.0050,0.5018


**Finding:** The odds ratio for state GDP is 0.9974, with a 95% confidence interval of [0.9898, 1.0050]. This suggests that a $100 billion increase in total state GDP is associated with only a very small estimated decrease in the odds of default, holding the other variables constant. However, the effect is not statistically significant (p = 0.502), and the confidence interval includes 1. Therefore, the analysis does not provide evidence that total state GDP meaningfully changes default odds after controlling for traditional borrower and loan characteristics.

## 8. Baseline vs. Augmented Model Comparison

To evaluate whether state GDP provides additional explanatory information, we compare the fit of the baseline and augmented logistic regression models.

Because the augmented model contains all baseline variables plus state GDP, the models can be directly compared using log-likelihood, Akaike Information Criterion (AIC), and a likelihood ratio test.

In [9]:
# Compare key model-fit statistics
model_comparison = pd.DataFrame({
    'Model': ['Baseline', 'Augmented + GDP'],
    'Log-Likelihood': [
        baseline_model.llf,
        augmented_model.llf
    ],
    'AIC': [
        baseline_model.aic,
        augmented_model.aic
    ],
    'Pseudo R-squared': [
        baseline_model.prsquared,
        augmented_model.prsquared
    ]
})

model_comparison.round(4)

,Model,Log-Likelihood,AIC,Pseudo R-squared
0,Baseline,-4470.5703,8955.1406,0.0613
1,Augmented + GDP,-4470.3441,8956.6881,0.0614


In [10]:
# Likelihood ratio test for whether adding state GDP
# significantly improves model fit
from scipy.stats import chi2

lr_stat = 2 * (
    augmented_model.llf - baseline_model.llf
)

df_difference = (
    augmented_model.df_model - baseline_model.df_model
)

lr_pvalue = chi2.sf(
    lr_stat,
    df_difference
)

print("Likelihood Ratio Statistic:", round(lr_stat, 4))
print("Degrees of Freedom:", int(df_difference))
print("P-value:", round(lr_pvalue, 4))

Likelihood Ratio Statistic: 0.4524
Degrees of Freedom: 1
P-value: 0.5012


**Finding:** Adding state GDP produces only a very small change in model fit. The Pseudo R-squared increases slightly from 0.0613 to 0.0614, while the AIC increases from 8955.14 to 8956.69. The likelihood ratio test is not statistically significant (LR = 0.4524, p = 0.5012). Therefore, adding total state GDP does not significantly improve the explanatory fit of the logistic regression model beyond the traditional borrower and loan characteristics.

## 9. Statistical Analysis Summary

The statistical analysis compared a baseline logistic regression using traditional borrower and loan characteristics with an augmented model that additionally included 2012 total state GDP.

The baseline results show that FICO score, DTI, annual income, loan amount, and loan term are significantly associated with default, while revolving utilization is not statistically significant after controlling for the other variables.

After state GDP is added, its coefficient is negative but not statistically significant (p = 0.502). Its odds ratio of 0.9974 is close to 1, and the 95% confidence interval includes 1, indicating little evidence of an independent association with default.

The augmented model also does not significantly improve overall model fit relative to the baseline model (likelihood ratio test p = 0.5012). Therefore, within this statistical specification, total state GDP does not provide significant additional explanatory information beyond the selected traditional borrower and loan characteristics.

These results do not determine whether GDP improves out-of-sample prediction. That question will be evaluated separately using machine learning models in Stage 4.